In [ ]:
import pandas as pd
import numpy as np
import os
import re
import glob
from datetime import datetime 
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill  
from openpyxl.utils import get_column_letter

# define today
today = datetime.now().strftime('%Y-%m-%d')

# set folder path for source files
folder_path = "C:/Users/kbixby/OneDrive - Northwell Health/Scripts/ots/source_files"
# get all xlsx files in the source folder
files = glob.glob(f'{folder_path}/*.xlsx')

def format_sheet(sheet):  
    # define styles
    header_font = Font(name='Ebrima', bold=True, color='000000', size=10)  
    cell_font = Font(name='Ebrima', size=10)  
    header_fill = PatternFill(start_color='BFBFBF', end_color='BFBFBF', fill_type='solid')  
    header_alignment = Alignment(horizontal='center', wrap_text=True)  
    cell_alignment = Alignment(horizontal='left')  
    thin_border = Border(  
        left=Side(style='thin'),  
        right=Side(style='thin'),  
        top=Side(style='thin'),  
        bottom=Side(style='thin')  
    )
    currency_format = '_($* #,##0_);_($* (#,##0);_($* "-"??_);_(@_)'
    integer_format = '###0'

    # format header row  
    for cell in sheet[1]:  
        cell.font = header_font  
        cell.fill = header_fill  
        cell.alignment = header_alignment  

    # format data rows  
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row, max_col=sheet.max_column):  
        for cell in row:  
            cell.font = cell_font
            cell.alignment = cell_alignment  
            if isinstance(cell.value, float):  
                cell.number_format = currency_format
            elif isinstance(cell.value, int):  
                cell.number_format = integer_format

    # auto-adjust column widths  
    for col in range(1, sheet.max_column + 1):  
        max_length = 0  
        column_letter = get_column_letter(col)  
        for row in range(1, sheet.max_row + 1):  
            cell = sheet.cell(row=row, column=col)  
            if cell.value:  
                max_length = max(max_length, len(str(cell.value)))  
        sheet.column_dimensions[column_letter].width = max_length + 4

    # add filter
    sheet.auto_filter.ref = sheet.dimensions

def add_total_row(sheet):  
    # get index for extra row
    last_row = sheet.max_row + 1  

    # for loop!
    for col in range(1, sheet.max_column + 1):  
        cell = sheet.cell(row=last_row, column=col)  
          
        # check if the column has numbers (skip first row header)  
        first_data_cell = sheet.cell(row=2, column=col)  

        if isinstance(first_data_cell.value, float):  
            # sum formula from row 2 to last data row  
            col_letter = get_column_letter(col)  
            cell.value = f'=SUM({col_letter}2:{col_letter}{last_row - 1})'  
        elif col == 1:  
            cell.value = 'Total'  
      
    # style the total row  
    total_font = Font(name='Ebrima', bold=True, size=10)  
    total_fill = PatternFill(start_color='D9D9D9', end_color='D9D9D9', fill_type='solid')  
    currency_format = '_($* #,##0_);_($* (#,##0);_($* "-"??_);_(@_)'  
    total_border = Border( 
        top=Side(style='thin'),  # double line above totals  
        bottom=Side(style='double')  
    )  
      
    for col in range(1, sheet.max_column + 1):  
        cell = sheet.cell(row=last_row, column=col)  
        cell.font = total_font  
        cell.fill = total_fill  
        cell.border = total_border  
        if isinstance(sheet.cell(row=2, column=col).value, float):  
            cell.number_format = currency_format  

def clean_ots_prepops(file, site):
    # create folder if needed
    os.makedirs(f'ots_summaries/{site}', exist_ok=True)

    # remove subtotals from file
    nosubs = file[file['account'].str.fullmatch(r'A_\d{5}', na=False)]
    # make the dept_id an integer
    nosubs['dept_id'] = nosubs['dept_id'].astype('int')

    # make numeric columns integers to prevent summing issues
    numeric_cols = ['2025_actual', '2026_budget', '2026_final_projection', '2027_budget_no_infl']  
    for col in numeric_cols:  
        nosubs[col] = pd.to_numeric(nosubs[col], errors='coerce')

    # group by department, sum columns
    dept_group = nosubs.groupby(['vp', 'director', 'dept_id', 'dept_name'], as_index = False)[['2025_actual', '2026_budget', '2026_final_projection', '2027_budget_no_infl']].sum()
    # rename columns for export
    dept_cap = dept_group.rename(columns={'vp': 'VP', 'director': 'Director', 'dept_id': 'Department ID', 'dept_name': 'Department', '2025_actual': '2025 Actual', 
                                             '2026_budget': '2026 Budget', '2026_final_projection': '2026 Final Projection', '2027_budget_no_infl': '2027 Budget w/o Inflation'})

    # group by account, sum columns
    acct_group = nosubs.groupby(['vp', 'account', 'acct_name'], as_index = False)[['2025_actual', '2026_budget', '2026_final_projection', '2027_budget_no_infl']].sum()
    # rename columns for export
    acct_cap = acct_group.rename(columns={'vp': 'VP', 'account': 'Account Number', 'acct_name': 'Account Name', '2025_actual': '2025 Actual', 
                                             '2026_budget': '2026 Budget', '2026_final_projection': '2026 Final Projection', '2027_budget_no_infl': '2027 Budget w/o Inflation'})
    # save all info to one sheet for reference
    with pd.ExcelWriter(f'ots_summaries/{site.upper()}_OTS_Summary_{today}.xlsx', engine='openpyxl') as writer:  
        dept_cap.to_excel(writer, sheet_name='Dept Summary', index=False)  
        acct_cap.to_excel(writer, sheet_name='Acct Summary', index=False)
        # format sheet
        for sheet in writer.sheets.values():
            format_sheet(sheet)
            add_total_row(sheet) 
            
    # print completed statement
    print(f"Created ots_summaries/{site}/{site}_OTS_Summary_{today}.xlsx")
    
    # collect vp names
    vps = set(dept_cap['VP'].unique()) & set(acct_cap['VP'].unique())

    # for loop!
    for vp in vps:  
        # filter dataframes to selected vp
        dept_group_vp = dept_cap[dept_cap['VP'] == vp]  
        acct_group_vp = acct_cap[acct_cap['VP'] == vp]  

        # clean name
        safe_name = re.sub(r'[^\w]', '_', vp)       # replaces ANY special character with _  
        safe_name = re.sub(r'_+', '_', safe_name)   # collapse multiple underscores into one  
        safe_name = safe_name.strip('_')            # remove leading/trailing underscores  

        # write to excel
        with pd.ExcelWriter(f'ots_summaries/{site}/{safe_name}_OTS_Summary_{today}.xlsx', engine='openpyxl') as writer:  
            dept_group_vp.to_excel(writer, sheet_name='Dept Summary', index=False)  
            acct_group_vp.to_excel(writer, sheet_name='Acct Summary', index=False)  

            # format sheets
            for sheet in writer.sheets.values():  
                format_sheet(sheet)
                add_total_row(sheet) 

        print(f"Created ots_summaries/{site}/{safe_name}_OTS_Summary_{today}.xlsx")  

# for loop!
for file in files:  
    # extract site name from file name
    filename = os.path.basename(file)
    site_name = os.path.splitext(filename)[0].split('_')[0].lower() 

    # read file     
    try:  
        df = pd.read_excel(file)  
        print(f"Processing {filename} → site: {site_name}")  
        clean_ots_prepops(df, site_name)  

    except PermissionError:  
        print(f"Skipping {filename} — close it in Excel and try again.") 

Processing GVCCC_OTS_Prepop_Clean.xlsx → site: gvccc
Created ots_summaries/gvccc/gvccc_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Busgith_Ray_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Nowierski_Paul_T_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Baker_Daniel_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Palmer_Demarkel_A_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Emergency_Events_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Stern_Robert_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Sartori_Danielle_E_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Cohen_Jill_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Russell_Jeffrey_C_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Dhir_Parul_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Jurik_Christopher_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Birnbaum_Laurie_OTS_Summary_2026-08-26.xlsx
Created ots_summaries/gvccc/Vi